# Logistic Regression and Model Evaluation – Avalanche Accidents in Switzerland

**Dataset:** All known avalanche accidents in Switzerland with at least one person caught, recorded by the WSL Institute
for Snow and Avalanche Research SLF (EnviDat, since 1970/71). We use the hydrological years 2000/01–2023/24.
File: `../00_Data/avalanche_accidents_switzerland.csv`.

| variable | meaning |
|:---|:---|
| `date`, `hydrological_year`, `month` | date of the accident (a hydrological year runs from October to September) |
| `canton`, `municipality`, `latitude`, `longitude` | location of the start zone |
| `elevation`, `aspect`, `inclination` | elevation (m a.s.l.), slope aspect (N, NE, …) and inclination (°) of the start zone |
| `danger_level` | avalanche danger level of the bulletin (1 = low … 5 = very high) |
| `activity` | `tour` (backcountry touring), `offpiste` (off-piste skiing), `transportation.corridor` (roads, railways), `building` |
| `caught`, `fully_buried`, `dead` | number of persons caught, completely buried and killed |
| `fatal` | 1 = at least one person died – **target** |

**Business question:** When an avalanche alarm comes in, the rescue coordination has to decide within minutes which
resources to send (helicopter with emergency doctor, avalanche dogs, …). How well can the probability of a **fatal**
outcome be estimated from the information available at that moment, and which **threshold** should trigger the maximum
alarm level?

**After this exercise you can ...**
- compute probabilities of a logistic regression model by hand,
- fit and interpret a logistic regression in scikit-learn (including categorical features),
- compute and interpret accuracy, precision, recall, F1 and FPR,
- choose a threshold based on the costs of false positives and false negatives,
- draw and interpret a ROC curve and the AUC,
- explain why the choice of features must match the information available at prediction time.

## Concept

The concepts behind this exercise (formulas and worked examples) are
explained in the notebook [logistic_regression_evaluation_concept.ipynb](logistic_regression_evaluation_concept.ipynb). Read it before you start.

## Didaktischer Arbeitsmodus (vereinfacht)

- **Ziel:** Konzepte aus dem Slideset in kleinen Schritten umsetzen.
- **Arbeitsablauf:** (1) Aufgabe in eigenen Worten, (2) Einzelfall rechnen, (3) erst dann verallgemeinern (ggf. Schleife), (4) Zwischenergebnis prüfen, (5) Ergebnis in 1–2 Sätzen interpretieren.
- **Bei komplexen Aufgaben:** Wenn eine Schleife verlangt wird, zuerst genau einen Wert/eine Konfiguration bearbeiten und die Ausgabe prüfen – erst danach die Schleife ergänzen.
- **Hinweis:** Rollen von Konzept, Übung und Lösung unverändert lassen.



## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
)

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand
Mini-Ziel: Bearbeite zuerst nur Teil a), prüfe Ergebnis, dann Teil b) usw.

**Didaktische Zerlegung:**
1. Teil a) zuerst vollständig lösen.
2. Einen Einzelfall rechnen und Ausgabe prüfen.
3. Erst danach auf mehrere Werte/Modelle verallgemeinern.
4. Ergebnis in 1-2 Sätzen fachlich interpretieren.


*Use the notebook cell below as your calculator: type the formulas with the numbers in
Python (e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

**a) Probabilities.** A logistic regression with the only feature *number of completely buried persons* has the
coefficients $\beta_0 = -3.5$ and $\beta_1 = 2.5$. Compute the score $\beta_0 + \beta_1 x$ and the probability
$p = 1/(1+e^{-\text{score}})$ of a fatal outcome for 0, 1 and 2 completely buried persons (hint: `np.exp()`). Which
accidents are classified as fatal with the threshold $\tau = 0.5$?

**b) Metrics.** A model is applied to 1'000 accidents, 130 of which were fatal. It flags 160 accidents as fatal, of
which 100 really were. Fill in the confusion matrix (TP, FP, FN, TN) and compute accuracy, precision and recall.

**c) Baseline.** What accuracy does a "model" reach that simply predicts "not fatal" for every accident? What is its
recall?

In [ ]:
# a) Coefficients and 0, 1 and 2 completely buried persons
beta_0 = -3.5
beta_1 = 2.5
buried = np.array([0, 1, 2])
score = ___
p = ___
print('probability:', p.round(3))

# b) Confusion matrix
TP = ___
FP = ___
FN = ___
TN = ___

# Accuracy, precision and recall
# Your code here

# c) Baseline: always predict 'not fatal'
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 2: Import and explore the data
Mini-Ziel: Bearbeite zuerst nur Teil a), prüfe Ergebnis, dann Teil b) usw.

**Didaktische Zerlegung:**
1. Teil a) zuerst vollständig lösen.
2. Einen Einzelfall rechnen und Ausgabe prüfen.
3. Erst danach auf mehrere Werte/Modelle verallgemeinern.
4. Ergebnis in 1-2 Sätzen fachlich interpretieren.


a) Import the data and keep the hydrological years 2000/01 to 2023/24 (`hydrological_year >= '2000/01'`). How many
accidents remain? What share of them was fatal?

b) Plot the share of fatal accidents by `danger_level`, by `activity` and by the number of completely buried persons
(`fully_buried`, grouped into 0, 1, 2, 3+). Which variable separates fatal and non-fatal accidents best?

In [ ]:
# a) Import the data as aval, keep the years 2000/01 to 2023/24 and print the
#    number of accidents and the share of fatal accidents
#    File: '../00_Data/avalanche_accidents_switzerland.csv'
# Your code here

# b) Share of fatal accidents by danger level, activity and number of
#    buried persons
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 3: Logistic regression with one feature
Mini-Ziel: Bearbeite zuerst nur Teil a), prüfe Ergebnis, dann Teil b) usw.

**Didaktische Zerlegung:**
1. Teil a) zuerst vollständig lösen.
2. Einen Einzelfall rechnen und Ausgabe prüfen.
3. Erst danach auf mehrere Werte/Modelle verallgemeinern.
4. Ergebnis in 1-2 Sätzen fachlich interpretieren.


a) Fit a logistic regression `fatal ~ fully_buried` on **all** accidents of Exercise 2 (no split yet). Print
$\hat\beta_0$ and $\hat\beta_1$ and compare them with the coefficients of Exercise 1a.

b) Plot the predicted probability against `fully_buried` (0–8) together with the observed share of fatal accidents per
value (dots). Interpret the S-curve.

In [ ]:
# a) Fit the logistic regression fatal ~ fully_buried on all accidents
#    and print beta_0 and beta_1
log_reg_1 = LogisticRegression()
# Your code here

# b) Plot the predicted probability (0 to 8 buried persons) and the observed
#    share of fatal accidents
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 4: Which information is available when?
Mini-Ziel: Bearbeite zuerst nur Teil a), prüfe Ergebnis, dann Teil b) usw.

**Didaktische Zerlegung:**
1. Teil a) zuerst vollständig lösen.
2. Einen Einzelfall rechnen und Ausgabe prüfen.
3. Erst danach auf mehrere Werte/Modelle verallgemeinern.
4. Ergebnis in 1-2 Sätzen fachlich interpretieren.


For a model to be useful, it may only use information that is **available at the time of the prediction**. We compare
two feature sets:

| feature set | features | situation |
|:---|:---|:---|
| **A – terrain and bulletin** | `danger_level`, `elevation`, `aspect`, `activity`, `month`, `caught` | what is known about the place, the conditions and the group – but not how many were buried |
| **B – alarm with burial information** | set A + `fully_buried` | the alarm call reports how many persons are completely buried |

(`dead` must of course never be used – it *is* the outcome.)

a) Keep only accidents without missing values in these features. Create dummy variables for `aspect` and `activity`
(`pd.get_dummies(..., drop_first=True, dtype=int)`) and split the data into a stratified training (80 %) and test set
(20 %) with `random_state=42`.

b) Fit a logistic regression (with `StandardScaler`, in a `make_pipeline`) for both feature sets.

c) Plot the coefficients of model B. Which features increase the probability of a fatal outcome?

In [ ]:
# a) Complete cases
features_a = [
    'danger_level',
    'elevation',
    'aspect',
    'activity',
    'month',
    'caught',
]
features_b = features_a + ['fully_buried']
data = aval.dropna(subset=features_b)

# Dummy variables for aspect and activity
X = pd.get_dummies(
    data[features_b], columns=['aspect', 'activity'], drop_first=True, dtype=int
)
y = data['fatal']

# Stratified split: same share of fatal accidents in both sets
X_train, X_test, y_train, y_test = ___

# Feature sets: model A without and model B with fully_buried
cols_b = list(X.columns)
cols_a = list(X.columns)
cols_a.remove('fully_buried')

# b) Logistic regression (with standardised features) for both feature sets
model_a = ___
model_b = ___
# Your code here

# c) Coefficients of model B
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 5: Evaluation with the default threshold τ = 0.5
Mini-Ziel: Bearbeite zuerst nur Teil a), prüfe Ergebnis, dann Teil b) usw.

**Didaktische Zerlegung:**
1. Teil a) zuerst vollständig lösen.
2. Einen Einzelfall rechnen und Ausgabe prüfen.
3. Erst danach auf mehrere Werte/Modelle verallgemeinern.
4. Ergebnis in 1-2 Sätzen fachlich interpretieren.


a) Predict the test set with model B and `predict()` (= threshold 0.5). Print the confusion matrix
(`confusion_matrix(y_true, y_pred)`: rows = actual class, columns = predicted class) and compute accuracy, precision,
recall, F1 and FPR.

b) Compare the accuracy with the baseline "never fatal". Is the model useful for the rescue coordination?

In [ ]:
# a) Predictions of model B with the default threshold 0.5
y_pred = model_b.predict(X_test[cols_b])

# Confusion matrix and metrics
# Your code here

# b) Baseline accuracy (never fatal)
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 6: Choosing the threshold based on costs
Mini-Ziel: Bearbeite zuerst nur Teil a), prüfe Ergebnis, dann Teil b) usw.

**Didaktische Zerlegung:**
1. Teil a) zuerst vollständig lösen.
2. Einen Einzelfall rechnen und Ausgabe prüfen.
3. Erst danach auf mehrere Werte/Modelle verallgemeinern.
4. Ergebnis in 1-2 Sätzen fachlich interpretieren.


The rescue coordination estimates:
- **underestimating** a fatal accident (FN: no doctor and no maximum alarm although a person is in mortal danger) is
  about **10 times** as bad as
- **overestimating** a non-fatal accident (FP: maximum alarm and additional resources that turn out to be unnecessary).

a) Compute precision, recall, FPR and the total costs $10\cdot FN + 1\cdot FP$ on the test set for the thresholds
$\tau = 0.05, 0.10, \dots, 0.90$ (use `predict_proba` of model B). Show the results as a table and plot the costs
against $\tau$.

b) Which threshold minimises the costs? How do recall and FPR change compared to $\tau = 0.5$?

*Note: strictly speaking, the threshold should be chosen on validation data (or with CV); here we use the test set to keep
the exercise short.*

In [ ]:
y_prob_b = model_b.predict_proba(X_test[cols_b])[:, 1]
cost_fn = 10
cost_fp = 1

# a) Metrics and costs for the thresholds 0.05, 0.10, ..., 0.90
rows = []
# Mini-Schritt: zuerst einen einzelnen Fall rechnen/pruefen, dann die Schleife laufen lassen.
y_pred_tau = (y_prob_b >= tau).astype(int)
# Your code here (append a dictionary with tau, precision, recall, FPR
# and costs to rows)
thresholds = pd.DataFrame(rows)

# b) Cost-optimal threshold (store the best row as best) and figure of the
#    costs by threshold
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

## Exercise 7: ROC curve and AUC – model A vs. model B
Mini-Ziel: Bearbeite zuerst nur Teil a), prüfe Ergebnis, dann Teil b) usw.

**Didaktische Zerlegung:**
1. Teil a) zuerst vollständig lösen.
2. Einen Einzelfall rechnen und Ausgabe prüfen.
3. Erst danach auf mehrere Werte/Modelle verallgemeinern.
4. Ergebnis in 1-2 Sätzen fachlich interpretieren.


a) Plot the ROC curves of model A and model B on the test set and compute their AUC. Mark the points for $\tau = 0.5$
and for the cost-optimal threshold of model B.

b) Check the interpretation of the AUC empirically for model B: draw 100'000 random pairs (one fatal, one non-fatal
accident) from the test set and compute the share of pairs in which the fatal accident has the higher predicted
probability (ties count ½).

c) Interpret the AUC values. What does the comparison of model A and model B tell us?

In [ ]:
# a) ROC curves and AUC of both models
# Your code here

# b) Draw 100'000 random pairs (one fatal, one non-fatal accident)
rng = np.random.default_rng(42)
p_pos = y_prob_b[y_test == 1]
p_neg = y_prob_b[y_test == 0]
# Your code here

---

> ✏️ **Your answer**
>
> *Double-click this cell and write your answer here.*

---

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')